# SSM-DAN: run one experiment (LOSO)

Works on Kaggle and Colab (detected automatically); needs a GPU and Internet access.
On a workstation or server, use the same launcher from a terminal: `python -m ssm_dan.launch --help`.

* Methods and variants: `configs/methods/*.yaml` (`ssm_dan` variants: `no_ra`, `no_da`,
  `per_session_ra`, `source_only`, `source_only_head`, `no_car`; combine with commas, e.g. `no_car,no_ra`).
* Output: `results/<dataset>/<run>/seed<k>/` (Kaggle: `/kaggle/working`, Colab: `/content`).
* Data: read from `data/` in the repository; files already there are used as they are, missing ones are linked
  from the attached inputs, then fetched or downloaded. Exact files and folders: `data/README.md`.
* Session ended before all folds finished: run again with the earlier results visible
  (Kaggle: *Add Input -> Your Work -> Notebook Output*; Colab: earlier results in the mounted Drive).
  Finished folds are skipped.

In [ ]:
METHOD = 'ssm_dan'      # ssm_dan | eegconformer | atcnet | tcformer | eegencoder | satransnet | rmetnet | ttime | bft
VARIANT = ''            # '' = the method as in the paper; e.g. 'no_ra' or 'per_session_ra'
DATASET = 'bcic2a'      # bcic2a | bcic2b | zhou2016 | openbmi
SUBJECTS = 'all'        # 'all' | '3' | '1-18' | '1,4,7-9'
SEED = 0
GPUS = [0, 1]           # two GPUs (Kaggle T4 x2); one GPU: [0]
MODE = 'paper'          # 'paper': launch folds as the reported run did (all folds in one process on GPU 0,
                        # or one process per fold over GPUS for OpenBMI / per_session_ra);
                        # 'per_fold': one fold per GPU at a time (faster); 'one_process'
OVERRIDES = []          # e.g. ['max_epochs=2'] for a quick check, ['preprocessing.batch_size=32']
BRANCH = 'main'

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path('/tmp/ssm-dan')                 # outside the saved output
URLS = ['https://github.com/CuongDM1806/ssm-dan.git',
        'https://github.com/CuongDM1806/tcformer-test.git']   # former name of the repository
for url in URLS:
    if (REPO / '.git').exists():
        break
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, url, str(REPO)],
                   env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
subprocess.run(['git', '-C', str(REPO), 'log', '-1', '--oneline'], check=True)
sys.path.insert(0, str(REPO))
from ssm_dan import launch

launch.run(METHOD, DATASET, variant=VARIANT, subjects=SUBJECTS, seed=SEED, gpus=GPUS, mode=MODE,
           overrides=OVERRIDES)